# 21. Generation Anatomy（spec §16）

## 学習目標
- 生成を1トークンずつ観察する（確信度・エントロピー・top候補）
- 同一 logits に対し sampling 手法（greedy/temperature/top-k/top-p）だけを変えた差を見る
- 多様性指標（repetition率・distinct-n）で「反復」を定量化する

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
L_RUN = ROOT / "experiments/runs/m4_model_l_modern_seed42"
M5 = ROOT / "experiments/analysis_m5"

In [2]:
ga = load_json(M5 / "generation_anatomy.json")
rows = ga["steps"][:15]
df = pd.DataFrame([{
    "step": r["index"], "chosen": r["chosen"], "prob": r["chosen_prob"], "entropy": r["entropy"],
    "top5": ", ".join(f'{t}:{p}' for t,p in r["top5"][:3])
} for r in rows])
print(f"prompt: {ga['prompt']} （greedy 生成の各ステップ）")
df

prompt: 日本の首都は （greedy 生成の各ステップ）


,step,chosen,prob,entropy,top5
0,0,日本,0.0199,6.865,"日本:0.0199, 19:0.0118, 中国:0.0115"
1,1,に,0.0797,4.826,"に:0.0797, を:0.0624, で:0.0581"
2,2,帰,0.0523,5.941,"帰:0.0523, 本:0.0328, 比:0.0302"
3,3,国,0.3009,3.114,"国:0.3009, 属:0.1699, って:0.0857"
4,4,する,0.1990,3.316,"する:0.199, した:0.1508, し、:0.11"
5,5,予定,0.0369,5.821,"予定:0.0369, が、:0.0285, まで:0.0284"
6,6,。,0.1198,3.717,"。:0.1198, だ。:0.1137, だ:0.0948"
7,7,\n,0.3867,4.545,"\n:0.3867, 日本:0.039, その:0.0139"
8,8,日本,0.0432,6.507,"日本:0.0432, - :0.0362, 「:0.0261"
9,9,は,0.1445,4.462,"は:0.1445, に:0.0826, 人:0.0549"


In [3]:
# step ごとの確信度とエントロピー
steps = ga["steps"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=[s["index"] for s in steps], y=[s["chosen_prob"] for s in steps],
                         name="選択トークン確率", line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=[s["index"] for s in steps], y=[s["entropy"] for s in steps],
                         name="エントロピー(nats)", yaxis="y2", line=dict(color="#ff7f0e")))
fig.update_layout(title="生成ステップごとの確信度とエントロピー", template="plotly_white", height=380,
                  xaxis_title="生成ステップ", yaxis=dict(title="P(選択)"),
                  yaxis2=dict(title="entropy", overlaying="y", side="right"))
fig.show()

In [4]:
# sampling 手法スイープ: 多様性 vs 反復
sweep = ga["sampling_sweep"]
names = list(sweep)
fig = go.Figure()
fig.add_trace(go.Bar(x=names, y=[sweep[n]["metrics"]["repetition_rate"] for n in names], name="repetition率", marker_color="#d62728"))
fig.add_trace(go.Bar(x=names, y=[sweep[n]["metrics"]["distinct_2"] for n in names], name="distinct-2", marker_color="#2ca02c"))
fig.update_layout(barmode="group", title="sampling 手法別の反復率と多様性（同一モデル・同一prompt）",
                  template="plotly_white", height=380)
fig.show()
for n in ["greedy","temp0.7","temp1.5","top_p0.9"]:
    print(f'[{n}] {sweep[n]["sample"][:80]}')

[greedy] 日本に帰国する予定。
日本は日本に帰国する予定。
日本は日本に帰国する予定。
日本は日本に帰国する予定。
日本は日本に帰国する予定。
日本は日本に帰国する予定。
[temp0.7] 5月12日と、アジアの首都を拠点とするアジアを中心と行う中で、日本が台湾の首都圏を中心とした中国国内の首都圏を中心、台湾の首都圏を中心としてアジアが成長してきた
[temp1.5] ビル縦丁系と、アトリを利用して蛙書連を第一原馬面へ歩行していた。その後早くパソコンを出発売域域タイ船を使って配肌線体裁状態になっていました……3、４！ 浅き有学
[top_p0.9] ビル・菓子屋に変身。特に書店を経営するユダヤ政府が1973年に創立し、ジョージア州の作家である夫大島（アメリカ・ルール・ジュニアン州）の請け合いであるヨークの書


## Observation / Interpretation / Caveat
- **Observation**: greedy は反復率が高く distinct-2 が低い（ループ）。温度を上げ、top-k/top-p を使うと反復が下がり多様性が上がるが、高温では意味崩壊も増える。エントロピーが高いステップで「分岐」が起きている。
- **Interpretation**: 反復は決定則（greedy）とモデルの過信の相互作用で生じる小型LMの典型failure mode。sampling は同じ logits の**読み方**を変えるだけで、モデルの知識は変わらない。温度は多様性と一貫性の trade-off ノブ。
- **Caveat**: distinct-n が低い=悪い、ではない（箇条書きや定型文は正当に低い）。多様性指標は品質の代理にすぎない。1プロンプト系列の観察。

**次へ**: [22_memorization_analysis](22_memorization_analysis.ipynb)